# LangChain 실험 플레이그라운드 — 사용법

밈/신조어 키워드에 대해 질문 -> 벡터 검색(dense/sparse/융합) -> 프롬프트 조립 -> LLM 답변까지,
각 셀을 독립적으로 재실행하며 실험하기 위한 노트북입니다.

## 실행 전 준비

이 노트북은 기본적으로 **EC2 서버(진짜 운영 데이터)** 를 봅니다. 그러려면 별도 터미널에서 SSH 터널이 먼저 떠 있어야 합니다:
```
ssh -i ~/.ssh/id_ed25519 -L 27017:localhost:27017 -L 36333:localhost:6333 ec2-user@100.29.36.216
```
(mongo는 27017 그대로, qdrant는 EC2용으로 `36333`을 씀 — `6333`은 Windows 예약 포트라 로컬에서 못 열고, `16333`은 로컬 docker qdrant가 이미 쓰고 있어서 겹치지 않게 분리했습니다.)

## 실행 순서

1번(환경설정) → 2번(키워드 선택) → 2.5번(유행 상태, 선택) → 3번(질문+임베딩) → 4번(검색) → 5번(검색결과 확인) → 6번(프롬프트) → 7번(LLM 호출) 순서로 **한 번 끝까지** 실행하세요.

그다음부터는 **3번 ~ 7번만** 값을 바꿔가며 반복 재실행하면 됩니다 (1, 2번은 한 번만 실행하면 됨).

## 셀별로 바꿀 수 있는 값

| 셀 | 변수 | 용도 |
|---|---|---|
| 1. 환경설정 | `USE_EC2` | `True`=EC2 진짜 데이터(터널 필요), `False`=로컬 docker 테스트 데이터 |
| 2. 키워드 선택 | `KEYWORD` | 분석할 밈/신조어. 셀 실행하면 임베딩 완료된 키워드 목록이 출력됨 |
| 2.5. 유행 상태 | `INCLUDE_TREND` | 네이버 데이터랩 유행 상태를 프롬프트에 넣을지 여부 |
| 3. 질문+임베딩 | `QUESTION` | LLM에게 물어볼 자유 질문 |
| 4. 검색 파라미터 | `TOP_K` | 근거로 가져올 청크 개수 |
| 4. 검색 파라미터 | `SOURCES` | 검색 대상 소스 제한. 예: `["dcinside","natepann","namuwiki"]`로 tavily/youtube 제외. `None`=전체 |
| 6. 프롬프트 작성 | `PROMPT_TEMPLATE` | LLM에게 보낼 지시문 자체를 수정 (`{keyword}`/`{trend_info}`/`{context}`/`{question}` 자리표시자는 유지) |
| 7. LLM 호출 | `MODEL`/`TEMPERATURE`/`TOP_P` | 같은 프롬프트로 답변이 어떻게 달라지는지 비교 |

## 5번 셀(검색결과 확인) 읽는 법

- `[D]`/`[S]`/`[DS]`: FUSED 결과가 dense 검색에서 왔는지, sparse에서 왔는지, 둘 다인지
- `⚠️본문에 키워드 없음`: 그 청크에 `KEYWORD`가 실제로 안 들어있다는 뜻 — **크롤링 오염 의심** (아래 참고)

## 알려진 이슈

일부 키워드(특히 tavily/youtube 소스)는 크롤링 때 관련 없는 글이 섞여 들어가 있습니다 (예: "야르" 검색에 "골반통신" 밈 설명 글이 딸려온 사례). `SOURCES`로 특정 소스를 빼고 비교해보면 어느 소스가 원인인지 확인할 수 있지만, 소스를 통째로 빼면 그 소스의 멀쩡한 자료까지 같이 빠지는 트레이드오프가 있습니다 — 아직 문서 단위로 걸러내는 근본 해법은 적용 전입니다.

## 1. 환경 설정

**이 셀이 하는 일**: OS 환경변수, 인코딩, import를 준비합니다.

**바꿀 것**: `USE_EC2` — EC2 진짜 데이터를 보려면 `True`(기본값), 로컬 docker에 테스트 데이터를 직접 채웠을 때만 `False`.

**주의**:
- `.env`의 `MONGODB_URI`/`QDRANT_HOST`는 `mongo`/`qdrant`라는 docker-compose 내부 네트워크 호스트명을 가리키며 `mimori-flask` 컨테이너 안에서만 resolve됩니다. `.env` 파일 자체는 건드리지 마세요(도커 컨테이너들이 깨집니다) — 이 셀이 로컬 커널일 때 자동으로 `localhost`로 대체합니다.
- `USE_EC2=True`로 쓰려면 아래 SSH 터널이 먼저 떠 있어야 합니다:
  ```
  ssh -i ~/.ssh/id_ed25519 -L 27017:localhost:27017 -L 36333:localhost:6333 ec2-user@100.29.36.216
  ```
  (mongo는 27017 그대로, qdrant는 EC2용으로 36333을 씀. `6333`은 Windows가 예약한 포트 범위라 로컬에서 못 열고, `16333`은 로컬 docker qdrant가 이미 쓰고 있어서 겹치지 않게 `36333`을 씀.)

In [1]:
import sys
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# 이 커널이 docker 컨테이너 밖(로컬)에서 실행 중이면 .env의 mongo/qdrant 호스트명이
# DNS로 안 풀립니다. 컨테이너 안(env_file로 이미 mongo/qdrant가 주입된 상태)에서 실행
# 중이면 아래 setdefault는 아무 효과가 없고, 로컬 커널일 때만 값이 대체됩니다.
#
# USE_EC2 = True: EC2 서버(진짜 운영 데이터)에 SSH 터널로 붙습니다.
#   ssh -i ~/.ssh/id_ed25519 -L 27017:localhost:27017 -L 36333:localhost:6333 ec2-user@100.29.36.216
#   (mongo는 27017 그대로, qdrant는 EC2용으로 36333을 씀 — 로컬 docker qdrant가 16333을 이미
#    쓰고 있고, 6333 자체는 Windows가 예약한 포트 범위라 로컬에서 못 엽니다.)
# USE_EC2 = False: 로컬 docker-compose qdrant(16333)를 봅니다. 로컬에서 직접 크롤링/전처리/
#   임베딩을 돌려서 테스트 데이터를 채웠을 때만 씁니다.
USE_EC2 = True

os.environ.setdefault("MONGODB_URI", "mongodb://localhost:27017")
os.environ.setdefault("QDRANT_HOST", "localhost")
os.environ["QDRANT_PORT"] = "36333" if USE_EC2 else "16333"

# 이 노트북은 analysis/ 안에 있으므로 커널 작업 디렉토리는 analysis/ 입니다.
# 프로젝트 루트(analysis/의 상위 폴더)를 import 경로에 추가합니다.
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

if sys.platform == "win32":
    try:
        sys.stdin.reconfigure(encoding="utf-8")
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except AttributeError:
        pass

from analysis.pipeline import list_analyzable_keywords
from analysis.rag_pipeline import (
    search_relevant_chunks,
    search_dense_only,
    search_sparse_only,
)
from embedding.encoder import encode_batch
from config.config_cilent import NIM_KEY
from langchain_nvidia_ai_endpoints import ChatNVIDIA

print("설정 완료 (EC2 모드)" if USE_EC2 else "설정 완료 (로컬 모드)")

C:\workspace\memeory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


설정 완료 (EC2 모드)


## 2. 키워드 선택

**이 셀이 하는 일**: MongoDB에서 임베딩이 끝난 밈 키워드 목록을 가져와 보여줍니다.

**실험하려면**: 출력된 목록 중 하나를 골라 `KEYWORD` 변수에 문자열로 직접 대입한 뒤 재실행하세요.

In [2]:
keywords = list_analyzable_keywords()
print(f"분석 가능한 키워드 {len(keywords)}개:")
for kw in keywords:
    print(f"  - {kw}")

KEYWORD = "거제야호"
print(f"\n현재 선택된 KEYWORD = {KEYWORD!r}")

분석 가능한 키워드 13개:
  - 거제야호
  - 럭키비키
  - 롤린
  - 리센느
  - 뭔말알
  - 쌰갈
  - 아자스
  - 야르
  - 야호~
  - 에스파
  - 영크크
  - 이순신
  - 홍명보

현재 선택된 KEYWORD = '거제야호'


## 2.5. z-score / 유행 상태 확인

**이 셀이 하는 일**: 선택한 키워드(`KEYWORD`)의 최근 검색량 기반 유행 상태를 네이버 데이터랩에서 조회합니다.

**실험하려면**: 아래 코드 셀의 `INCLUDE_TREND`를 `True`/`False`로 바꾼 뒤, 이 셀을 먼저 재실행하고 "6. 프롬프트 작성" 셀을 실행해야 반영됩니다(`INCLUDE_TREND`는 이 셀에서 정의되므로, 값만 바꾸고 이 셀을 다시 실행하지 않으면 이전 값이 그대로 남아있습니다).

In [3]:
from trend.trend_service import format_trend_context

trend_info = format_trend_context(KEYWORD)
if trend_info:
    print(trend_info)
else:
    print("z-score/유행 상태를 가져오지 못했습니다 (NAVER API 키 미설정이거나 데이터 없음).")

INCLUDE_TREND = True

[trend_service] 카카오 지표 수집 실패, 제외 진행: KAKAO_REST_API_KEY 가 설정되지 않았습니다. .env 를 확인하세요.
[google_client] 요청 실패(ModuleNotFoundError("No module named 'pytrends'")), 2초 후 재시도 (1/2)
[google_client] 요청 실패(ModuleNotFoundError("No module named 'pytrends'")), 4초 후 재시도 (2/2)
[google_client] 최대 재시도 초과, google_z 제외: ModuleNotFoundError("No module named 'pytrends'")
[참고: 최근 검색/언급량 기반 유행 상태 앙상블 판정 — 정성적 분석의 보조 지표로만 활용]
상태: 감소 (robust z-score: -1.17, 반영 소스: naver)


## 3. 질문 입력 + 임베딩

**이 셀이 하는 일**: 자유 텍스트 질문을 BGE-M3로 dense 벡터 + sparse(lexical) 벡터로 변환합니다.

**실험하려면**: `QUESTION` 문자열만 바꾸면 됩니다. 질문이 바뀌면 검색되는 청크가 달라지므로, 이 셀부터 다시 실행해야 아래 결과에 반영됩니다. (임베딩 모델은 재실행마다 새로 로드하지 않도록 그대로 유지합니다 — LLM 호출은 NVIDIA API라 로컬 GPU와 충돌하지 않습니다.)

In [4]:
QUESTION = "이 밈은 왜 유행했나요?"

dense_vecs, lexical_weights = encode_batch([QUESTION])
dense_vec, sparse = dense_vecs[0], lexical_weights[0]

print(f"QUESTION = {QUESTION!r}")
print(f"dense_vec 길이 = {len(dense_vec)}")
print(f"sparse 항목 수 = {len(sparse)}")

[임베딩] BAAI/bge-m3 로드 중... (device=cuda)


Loading weights: 100%|██████████| 391/391 [00:03<00:00, 113.24it/s]


QUESTION = '이 밈은 왜 유행했나요?'
dense_vec 길이 = 1024
sparse 항목 수 = 10


## 4. 검색 파라미터 + 실행

**이 셀이 하는 일**: 같은 질문 벡터로 dense만 / sparse만 / RRF 융합, 세 가지 방식으로 Qdrant에서 관련 청크를 검색합니다.

**실험하려면**:
- `TOP_K`를 늘리거나 줄여서 더 많은/적은 근거 청크를 가져와볼 수 있습니다.
- `SOURCES`로 검색 대상 소스를 제한할 수 있습니다. 예를 들어 tavily/youtube가 오염됐다고 의심되면 `SOURCES = ["dcinside", "natepann", "namuwiki"]`로 빼고 돌려서, 답변이 좋아지는지 비교해보세요. `None`이면 전체 소스(`tavily`/`youtube`/`namuwiki`/`natepann`/`dcinside`)를 다 봅니다.

In [5]:
TOP_K = 15
SOURCES = ["dcinside", "natepann", "namuwiki"]  # 예: ["dcinside", "natepann", "namuwiki"] 처럼 특정 소스만 검색할 수 있음 (None=전체)

dense_points = search_dense_only(KEYWORD, dense_vec, TOP_K, sources=SOURCES)
sparse_points = search_sparse_only(KEYWORD, sparse, TOP_K, sources=SOURCES)
fused_points = search_relevant_chunks(KEYWORD, dense_vec, sparse, TOP_K, sources=SOURCES)

print(f"SOURCES = {SOURCES!r}")
print(f"dense={len(dense_points)}, sparse={len(sparse_points)}, fused={len(fused_points)}")

SOURCES = ['dcinside', 'natepann', 'namuwiki']
dense=15, sparse=15, fused=15


## 5. 검색 결과 확인 ("왜 이게 뽑혔나" 진단 포함)

**이 셀이 하는 일**: dense / sparse / 융합(RRF) 검색 결과를 나란히 비교 출력합니다. 각 결과 옆에:
- `[D]`/`[S]`/`[DS]`: FUSED 결과가 dense에서만 왔는지, sparse에서만 왔는지, 둘 다에서 왔는지(RRF가 왜 이 청크를 올렸는지)
- `⚠️본문에 키워드 없음`: 이 청크의 본문/제목에 `KEYWORD` 문자열이 실제로 안 들어있으면 표시 — 크롤링 오염(다른 주제의 글이 이 키워드로 잘못 저장된 경우) 즉시 확인 가능

**실험하려면**: 이 셀 자체는 결과를 출력만 하므로 수정할 것이 없습니다 — 위 셀들(질문, TOP_K, 키워드)을 바꾸고 재실행해서 차이를 비교하세요. `⚠️` 표시가 많이 뜨면, 그 키워드는 크롤링 데이터 자체가 오염됐을 가능성이 높습니다 (실제로 `야르`에서 발견된 사례 — tavily가 "밈 뜻 유래" 같은 범용 검색어에 이끌려 전혀 다른 밈을 설명하는 글까지 가져와서 `야르`로 잘못 저장한 경우가 있었습니다).

In [6]:
def _print_points(label, points, dense_ids=None, sparse_ids=None):
    print(f"--- {label} ({len(points)}개) ---")
    for p in points:
        title = p.payload.get("title") or "제목 없음"
        url = p.payload.get("url") or "출처 없음"
        text = p.payload.get("text", "")

        origin = ""
        if dense_ids is not None and sparse_ids is not None:
            in_dense = p.id in dense_ids
            in_sparse = p.id in sparse_ids
            origin = "[" + ("D" if in_dense else "") + ("S" if in_sparse else "") + "] "

        contains_kw = KEYWORD in text or KEYWORD in title
        warn = "" if contains_kw else "  ⚠️본문에 키워드 없음(크롤링 오염 의심)"

        print(f"  {origin}score={p.score:.4f} | {title} ({url}){warn}")
        print(f"  {text[:150]}")
    print()

dense_ids = {p.id for p in dense_points}
sparse_ids = {p.id for p in sparse_points}

_print_points("DENSE", dense_points)
_print_points("SPARSE", sparse_points)
_print_points("FUSED(RRF)", fused_points, dense_ids, sparse_ids)

--- DENSE (15개) ---
  score=0.5173 | 우리나라 개그맨 예능인들 진짜 반성해야됨 (https://pann.nate.com/talk/375492830)
  거제야호 무야호 둘 다 엄청난 센세이션을 일으킨 유행어들이지 근데 웃긴게 저 유행어를 만든게 일본인 아이돌이랑
미국 사는 할아버지임 이게 맞는거냐? 하..
요즘 개그맨들 개그는 안하고 유행어도 하나 없고 뭔
쇼츠 생성용 개그나
틱톡 밈 이런거나 따라하고 있음
예능인들도
  score=0.4910 | 리센느 갑자기 뜬 벼락부자 아니야 (https://gall.dcinside.com/board/view/?id=drama_new3&no=23697697)  ⚠️본문에 키워드 없음(크롤링 오염 의심)
  자랑하는건 아닌데
나를 비롯해 하드코어 덕후들은 이미 알음알음 리센느 덕질하고 있었어
애들 예쁘고 포텐 있었고... 거제 야호는 그저 기폭제였을 뿐이고

[댓글]
얘 이름 먼데 존예네
  score=0.4783 | 작년에 럽어택 역주행 할 때 (https://gall.dcinside.com/mgallery/board/view/?id=rescene1&no=147575)
  우연히 쇼츠에 중소돌 기적이라고 캄서 떠서 들으니 노래가 너무 좋은 거야
글서 회사 아이돌 잘알 동생한테 물어보니 개차갑게 바이럴 너무 많다고 칼대답 하길래 머쓱타드 하고 말았던 나란 새끼..그때.갤이라도 와 볼 걸...(그랬으면 더 빨리 입덕했을 건데)
올해 거제야호
  score=0.4769 | 나는 리센느 팬이 아니야 (https://gall.dcinside.com/mgallery/board/view/?id=rescene1&no=147137)  ⚠️본문에 키워드 없음(크롤링 오염 의심)
  [댓글]
ㅋㅋㅋ
입덕부정기는 일단 버블 결제해놓고 시작해라
이미 구독했지
@카-미디언 버블까지 했지만 암튼 팬은 아닌거지? ㅋㅋㅋ
플챗 친구비는 냈음?
(흐뭇)
[MSI] BLG 양대인 감독, "'제우스' 강심장, 리스펙 하게 됐다"

## 6. 프롬프트 작성

**이 셀이 하는 일**: 검색된 청크(`fused_points`)와 유행 상태 정보(`trend_info`, `INCLUDE_TREND`가 `True`일 때만)를 근거 자료로 넣어 LLM에게 실제로 전달할 프롬프트를 완성합니다.

**실험하려면**: `PROMPT_TEMPLATE` 문자열을 자유롭게 수정하세요(지시문 추가, 어조 변경, 출력 형식 지정 등). `{keyword}` / `{trend_info}` / `{context}` / `{question}` 네 자리표시자는 반드시 그대로 남겨둬야 합니다. `.format()`을 쓰므로, 프롬프트에 `{`/`}` 자체를 문자로 넣고 싶으면(예: JSON 출력 형식을 지시하는 경우) `{{`/`}}`로 이스케이프해야 합니다.

In [7]:
PROMPT_TEMPLATE = """당신은 밈/신조어 분석 전문가입니다.

키워드: {keyword}

{trend_info}
자료:
{context}

질문: {question}

한국어로 답변하세요.위 자료를 근거로 답변하세요. 자료에 없는 내용은 추측하지 말고 모른다고 답하세요.
"""

context = "\n\n---\n\n".join(
    f"[출처: {p.payload.get('title') or '제목 없음'} / {p.payload.get('url') or '출처 없음'}]\n{p.payload.get('text', '')}"
    for p in fused_points
)
prompt = PROMPT_TEMPLATE.format(
    keyword=KEYWORD,
    trend_info=(trend_info if INCLUDE_TREND else ""),
    context=context,
    question=QUESTION,
)
print(prompt)

당신은 밈/신조어 분석 전문가입니다.

키워드: 거제야호

[참고: 최근 검색/언급량 기반 유행 상태 앙상블 판정 — 정성적 분석의 보조 지표로만 활용]
상태: 감소 (robust z-score: -1.17, 반영 소스: naver)
자료:
[출처: 우리나라 개그맨 예능인들 진짜 반성해야됨 / https://pann.nate.com/talk/375492830]
거제야호 무야호 둘 다 엄청난 센세이션을 일으킨 유행어들이지 근데 웃긴게 저 유행어를 만든게 일본인 아이돌이랑
미국 사는 할아버지임 이게 맞는거냐? 하..
요즘 개그맨들 개그는 안하고 유행어도 하나 없고 뭔
쇼츠 생성용 개그나
틱톡 밈 이런거나 따라하고 있음
예능인들도 마찬가지 재밌게 하거나 웃기려고 전혀 애쓰지를 않음 편한것만 추구하고 힘들고 어려운건 거부하고 배척하고 있음
과거 숱한 예능을 보고 울고 웃었던 시청자들이 현재 예능과 예능인들을 비판하고 있는게 현실임
예능이라는게 언짢으려 보는건 아니잖아?
언제 시청자들이 다시 예능을 보며 즐거워 할 날이 올 수 있을까 하..

---

[출처: 리센느 거제야호밈 완전 성공했다잉 / https://pann.nate.com/talk/375465875]
선배들이 말아주는 야호~ 밈 몰아보기
1.방탄소년단 선배님
진
뷔
정국
방탄 : 부산 야호 ~
(원작자인 미나미가 매우매우 좋아했음)
2.태양 선배님
태양 : 태양 야호 ~
(첼린지때 배려해주셔서 넘 감사했던 미나미)
아이돌쪽만 유행이었느냐? ㄴㄴ
3.전지현 선배님
(전지현이 선택한 바이오 문구 : 야호 ~)
전지현& 팀군체 : 군체 야호 ~
(원본으로 피날레 장식)
미나미 야호 ~ 리센느 야호 ~

---

[출처: 리센느 갑자기 뜬 벼락부자 아니야 / https://gall.dcinside.com/board/view/?id=drama_new3&no=23697697]
자랑하는건 아닌데
나를 비롯해 하드코어 덕후들은 이미 알음알음 리센느 덕질하고 있었어
애들 예쁘고 포텐 있었고... 거제 야호는 그저 기폭제였

## 7. LLM 호출

**이 셀이 하는 일**: 완성된 프롬프트를 NVIDIA API(ChatNVIDIA)로 보내고 답변을 받아 출력합니다.

**실험하려면**: `MODEL` / `TEMPERATURE` / `TOP_P` 값을 바꿔서, 같은 프롬프트에도 답변이 어떻게 달라지는지 비교해보세요.

In [ ]:
assert NIM_KEY, "NIM_KEY가 .env에 설정되어 있지 않습니다"

from analysis.pipeline import invoke_with_retry

MODEL = "deepseek-ai/deepseek-v4-flash"
TEMPERATURE = 1
TOP_P = 0.95

llm_client = ChatNVIDIA(
    model=MODEL,
    api_key=NIM_KEY,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_completion_tokens=16384,
    timeout=6000,
)

# 503(서버 혼잡) 등 일시적 오류는 지수 백오프로 자동 재시도하고 진행 상황을 출력한다.
response = invoke_with_retry(llm_client, [{"role": "user", "content": prompt}])
print(response.content)